In [26]:
#Libraries needed

import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
from IPython.display import clear_output
import torch
from torch import nn
from torch.utils.data import DataLoader , Dataset

In [27]:
rand = np.random.default_rng(42)

#Defining the values we talked about
a = -10
b = 10
n = 1000

#Our target function f
def f(x,y):
    return x**2+y**2


xcoords = rand.uniform(low=a,high=b,size=(n,1))
ycoords = rand.uniform(low=a,high=b,size=(n,1))

zcoords = f(xcoords,ycoords)

dataset = np.concatenate((xcoords,ycoords,zcoords),axis=1)

#formatting coords for input
TestTrainSplit = 0.8
batches = 2
SplitIndex = int(TestTrainSplit*n)
TrainData  = dataset[:SplitIndex]
TestData = dataset[SplitIndex:]

noise = rand.normal(loc=0,scale=0.5,size=(SplitIndex,1))
TrainData[:,-1] = (TrainData[:,-1].reshape(SplitIndex,1) + noise).reshape(SplitIndex)

TestData = torch.from_numpy(TestData).to(dtype=torch.float32)
TrainData = torch.from_numpy(TrainData).to(dtype=torch.float32)

#Simple scatter plot for visualisation
fig = px.scatter_3d(TrainData,x=0,y=1,z=2)
fig.show()

In [28]:
batch_num = 10

class DataFormat(Dataset):
    def __init__(self,data):
        self.input_shape = (2,1)
        self.inputs = data[:,:-1]
        self.labels = data[:,-1].reshape(data[:,-1].shape[0],1)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, index):
        current_input = self.inputs[index].reshape(self.input_shape)
        current_label = self.labels[index]
        return current_input , current_label



Train_Data = DataFormat(TrainData)
Test_Data = DataFormat(TestData)

Train_Batches = DataLoader(Train_Data,batch_size=batch_num)
Test_Batches = DataLoader(Test_Data,batch_size=batch_num)



for X, y in Test_Batches:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([10, 2, 1])
Shape of y: torch.Size([10, 1]) torch.float32


In [29]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(2, 300),
            nn.ReLU(),
            nn.Linear(300, 300),
            nn.ReLU(),
            nn.Linear(300, 300),
            nn.ReLU(),
            nn.Linear(300,1),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=2, out_features=300, bias=True)
    (1): ReLU()
    (2): Linear(in_features=300, out_features=300, bias=True)
    (3): ReLU()
    (4): Linear(in_features=300, out_features=300, bias=True)
    (5): ReLU()
    (6): Linear(in_features=300, out_features=1, bias=True)
  )
)


In [30]:
loss_func = nn.MSELoss()
optimiser = torch.optim.SGD(model.parameters(),lr=1e-4)


In [31]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [32]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [33]:
epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(Train_Batches, model, loss_func, optimiser)
    test(Test_Batches, model, loss_func)
print("Done!")

Epoch 1
-------------------------------
loss: 5879.923340  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 627.441463 

Epoch 2
-------------------------------
loss: 279.397034  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 262.328461 

Epoch 3
-------------------------------
loss: 101.336555  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 205.153950 

Epoch 4
-------------------------------
loss: 80.127365  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 177.113223 

Epoch 5
-------------------------------
loss: 74.089066  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 154.589419 

Epoch 6
-------------------------------
loss: 71.384232  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 135.558583 

Epoch 7
-------------------------------
loss: 68.977791  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 118.720847 

Epoch 8
-------------------------------
loss: 66.612228  [   10/  800]
Test Error: 
 Accuracy: 0.0%, Avg loss: 105.197027 

Epoc